# 개별종목 조합G — LogisticRegression

## 실험 목적

KOSPI200 방향이 상승·보합·하락 중 어디인지 정해졌을 때, 같은 방향일 확률이 높은
개별종목을 찾기 위한 3분류 모델입니다.

후보는 매 거래일 **KOSPI 세부 업종지수 시가총액 상위 10개 × 업종별 KOSPI 보통주
시가총액 상위 5개**로 먼저 고정합니다. 업종의 미래 방향을 따로 예측하는 구조는 아닙니다.

## 공통 조건

| 항목 | 값 |
|---|---|
| 원천 | HF `full/daily_price_dev.parquet`, `full/index_price_dev.parquet` |
| 홀드아웃 | `20240901` 이후 접근 금지 |
| 라벨 | T일 판단 → T+1 `adj_open` 진입 → T+6 `adj_open` 평가, 종목 ±2% |
| 외부 검증 | 날짜 그룹 expanding 12폴드 |
| 최초 학습 | 750거래일 |
| 검증·gap | 폴드당 60거래일 · 직전 5거래일 제거 |
| class weight | 각 외부 폴드 내부에서 `None`과 `balanced` 재비교 |
| 선정 지표 | Accuracy·Macro F1·하락 Recall 조화평균 |

## OOS 결과

| Accuracy | Macro F1 | 하락 Recall | 핵심지표 조화평균 |
|---:|---:|---:|---:|
| 0.4107 | 0.3676 | 0.2401 | **0.3130** |

아래 셀은 저장된 실측 리포트에서 이 모델의 폴드 결과와 class weight 선택 횟수를 다시
읽습니다. 학습 구현은 `models/stock_experiment.py`, 피처·라벨은
`features/stock_model_dataset.py`가 정본입니다.


In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "reports" / "stock_feature_combinations.json").exists():
    ROOT = ROOT.parent
report_path = ROOT / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination = 'G'
model_name = 'LogisticRegression'

combination_report = report["combinations"][combination]
folds = pd.DataFrame(combination_report["outer_fold_results"])
display(folds.loc[folds["model"].eq(model_name)].reset_index(drop=True))

weights = pd.DataFrame(combination_report["selected_class_weight_counts"])
display(weights.loc[weights["model"].eq(model_name)].reset_index(drop=True))


,model,fold,selected_class_weight,train_dates,valid_dates,train_rows,valid_rows,train_end,valid_start,valid_end,...,down_recall,core_harmonic_mean,training_majority_class,training_majority_baseline_accuracy,validation_majority_class,validation_majority_oracle_accuracy,validation_down_rate,validation_neutral_rate,validation_up_rate,accuracy_minus_training_majority_baseline
0,LogisticRegression,1,NaN,750,60,35611,2875,20140207,20140217,20140514,...,0.093704,0.188159,0,0.501217,0,0.501217,0.237565,0.501217,0.261217,-0.000348
1,LogisticRegression,2,balanced,980,60,46623,2785,20150115,20150123,20150421,...,0.187070,0.274221,0,0.397846,0,0.397846,0.261041,0.397846,0.341113,-0.015081
2,LogisticRegression,3,balanced,1210,60,57309,2919,20151217,20151228,20160328,...,0.215232,0.295904,0,0.376156,0,0.376156,0.310380,0.376156,0.313464,-0.002055
3,LogisticRegression,4,balanced,1439,60,68495,2848,20161124,20161202,20170228,...,0.190672,0.299938,0,0.461728,0,0.461728,0.255969,0.461728,0.282303,0.001053
4,LogisticRegression,5,balanced,1669,60,79206,2720,20171103,20171113,20180207,...,0.347575,0.385845,0,0.390074,0,0.390074,0.318382,0.390074,0.291544,0.031250
5,LogisticRegression,6,balanced,1899,60,89642,2776,20181016,20181024,20190118,...,0.300000,0.365918,0,0.372478,0,0.372478,0.291787,0.372478,0.335735,0.042507
6,LogisticRegression,7,balanced,2129,60,100686,2928,20190920,20190930,20191224,...,0.213873,0.310862,0,0.478142,0,0.478142,0.236339,0.478142,0.285519,-0.021175
7,LogisticRegression,8,balanced,2359,60,111867,2940,20200825,20200902,20201130,...,0.366337,0.381247,0,0.347619,1,0.377551,0.274830,0.347619,0.377551,0.042857
8,LogisticRegression,9,balanced,2589,60,123092,2916,20210729,20210806,20211105,...,0.166227,0.256351,0,0.391632,0,0.391632,0.389918,0.391632,0.218450,-0.030178
9,LogisticRegression,10,balanced,2818,60,134110,2898,20220706,20220714,20221012,...,0.211732,0.303898,0,0.345411,-1,0.376467,0.376467,0.345411,0.278123,0.047964


,model,selected_class_weight,folds
0,LogisticRegression,None,1
1,LogisticRegression,balanced,11
